In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import binascii

# Load up the sensor dataset we just created
filename = 'ESP32 Multisensor Dataset.csv'
df = pd.read_csv(filename)

# Scale the sensor data down to values between 0.0 and 1.0.
# We have to remember to do this exact same math in the Arduino code later!
df['Temperature'] = df['Temperature'] / 50.0
df['Humidity'] = df['Humidity'] / 100.0
df['Vibration'] = df['Vibration'] / 4095.0
df['Distance'] = df['Distance'] / 200.0

# Map our environment states to simple numbers (0, 1, 2, 3)
label_mapping = {
    'system idle': 0,
    'machine running': 1,
    'proximity alert': 2,
    'tamper detected': 3
}
df['Label'] = df['Label'].map(label_mapping)

# X is our sensor data, y is the answer (the label)
X = df[['Temperature', 'Humidity', 'Vibration', 'Distance']].values
y = df['Label'].values

# Build a small neural network. Keeping the layers small (16 and 8)
# so it actually fits inside the ESP32 memory.
model = Sequential([
    Dense(16, activation='relu', input_shape=(4,)),
    Dense(8, activation='relu'),
    Dense(4, activation='softmax') # 4 classes to predict
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("Training the model... this should be quick.")
model.fit(X, y, epochs=100, batch_size=8, validation_split=0.2)

print("\nCompressing into TensorFlow Lite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# This function turns the TFLite file into a huge array of C++ hex bytes
# so we can just include it as a header file in the Arduino IDE.
def convert_to_c_array(bytes_data):
    hex_data = binascii.hexlify(bytes_data).decode('utf-8')
    c_array = ', '.join(['0x' + hex_data[i:i+2] for i in range(0, len(hex_data), 2)])

    # Add line breaks every 12 bytes so it doesn't look like a mess in Arduino
    c_array_formatted = ""
    for i, byte in enumerate(c_array.split(', ')):
        c_array_formatted += byte + ", "
        if (i + 1) % 12 == 0:
            c_array_formatted += "\n    "

    return c_array_formatted.strip(', ')

c_code = f"""// Auto-generated TinyML model for ESP32 Practical
#ifndef MODEL_H
#define MODEL_H

// This is the actual compressed neural network
const unsigned char model_tflite[] = {{
    {convert_to_c_array(tflite_model)}
}};

const int model_tflite_len = {len(tflite_model)};

#endif // MODEL_H
"""

# Save it directly as a .h file
with open("model.h", "w") as f:
    f.write(c_code)

print("\nDone.")


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training the model... this should be quick.
Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3063 - loss: 1.3894 - val_accuracy: 0.4250 - val_loss: 1.3660
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5063 - loss: 1.3443 - val_accuracy: 0.4500 - val_loss: 1.3249
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5125 - loss: 1.3026 - val_accuracy: 0.4500 - val_loss: 1.2800
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5125 - loss: 1.2596 - val_accuracy: 0.4500 - val_loss: 1.2329
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5125 - loss: 1.2142 - val_accuracy: 0.4500 - val_loss: 1.1894
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5125 - loss: 1.1681 - val_accuracy: 0.4500 - val_loss: 1.1418
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5125 - loss: 1.1223 - val_accuracy: 0.4500 - val_loss: 1.0964
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5